# ML03 · 逻辑回归与分类

用逻辑回归（logistic regression）把连续的线性预测转成「几率与类别」，创建二分类（binary classification）最基本的直觉。

## 学习目标
- 理解 sigmoid 函数如何把任意实数压成 0 到 1 之间的几率输出。
- 能说明二分类（binary classification）与线性回归（linear regression）在「输出意义」上的差别。
- 用图直觉认识决策边界（decision boundary）如何把平面切成两类。
- 用直觉（非严格推导）理解交叉熵（cross-entropy）损失为什么适合分类。
- 会计算与解读准确率（accuracy），并知道它在类别不平衡时的盲点。
- 能用 sklearn 的 LogisticRegression 完成一个自造数据的二分类示范。

In [ ]:
# 概念：统一导入与绘图设置，让后面每个范例可直接画图且中文标题不乱码
import numpy as np
import matplotlib.pyplot as plt

# 设置中文字体，避免图上中文变成方框（不同系统可换成手边有的字体）
plt.rcParams["font.sans-serif"] = ["Microsoft JhengHei", "SimHei", "Arial Unicode MS"]
plt.rcParams["axes.unicode_minus"] = False   # 负号正常显示，避免被当成特殊字符

print("环境设置完成")

## 从回归到分类：为什么需要几率

二分类（binary classification）要的答案只有两种：是 / 否、属于 / 不属于，习惯用标签（label）0 与 1 表示。

线性回归（linear regression）输出的是任意实数，可能是 -3、也可能是 250，没办法直接当「是不是某一类」的答案。我们需要一个输出落在 0 到 1、可以解读成几率（probability）的模型。

为什么先看它的局限：唯有先看到「直线预测会冲出 0 到 1」这个问题，才懂得后面为什么要把输出压进几率区间。

In [ ]:
# 概念：分类标签只有 0 / 1，但直线预测会超出 0 到 1，无法当几率解读
# 造一份假数据：楼高（公尺）对应「是否为高层建筑」（1 是、0 否）
heights = np.array([8, 12, 18, 25, 33, 40, 52, 60], dtype=float)
is_highrise = np.array([0, 0, 0, 1, 1, 1, 1, 1])   # 与上面每笔楼高一一对应

# 用最小平方法硬拟一条直线（楼高 -> 标签），示范直线预测的问题
slope, intercept = np.polyfit(heights, is_highrise, deg=1)   # deg=1 即一次直线
line_pred = slope * heights + intercept                      # 直线在每个楼高的预测值

print("楼高:", heights)
print("真实标签:", is_highrise)
print("直线预测值:", np.round(line_pred, 2))
# 注意：直线预测会出现负数或大于 1 的值，根本不能当「几率」解读
print("预测最小值:", round(line_pred.min(), 2), " 预测最大值:", round(line_pred.max(), 2))

## sigmoid 函数与几率输出

sigmoid 函数（sigmoid function）把任意实数（这里称为线性分数 z）压进 0 到 1 之间，使输出可以解读成几率（probability output）。

公式：sigmoid(z) = 1 / (1 + e^(-z))

它的形状像一条平滑的 S：z 很大时趋近 1、很小时趋近 0、z = 0 时刚好是 0.5。得到几率后，用阈值（threshold）0.5 当分界：几率大于等于 0.5 判为类别 1，否则判为类别 0。

In [ ]:
# 概念：用 sigmoid 把线性分数压成几率，并画出 S 形曲线创建直觉
def sigmoid(z):
    return 1 / (1 + np.exp(-z))   # 对应公式 1 / (1 + e^(-z))，z 可为数组（矢量化）

z = np.linspace(-8, 8, 200)       # -8 到 8 取 200 个等距点当输入分数
plt.figure(figsize=(5, 3))
plt.plot(z, sigmoid(z))
plt.axhline(0.5, color="gray", linestyle="--")   # 0.5 是常用的判类阈值
plt.title("sigmoid 函数")
plt.xlabel("线性分数 z")
plt.ylabel("几率")
plt.show()

# 把自造的「楼高分数」喂进 sigmoid，看楼越高几率越接近 1
heights = np.array([8, 18, 33, 52], dtype=float)
scores = 0.15 * heights - 3.5     # 自订一个线性分数：楼越高分数越大
probs = sigmoid(scores)           # 转成「属于高层」的几率
for h, p in zip(heights, probs):
    print(f"楼高 {h:4.0f} 公尺 -> 几率 {p:.3f}")

## 决策边界：把平面切成两类

当有两个特征时，每个样本是特征空间（feature space）里的一个点。模型先算线性分数，再经 sigmoid 得到几率，最后用阈值 0.5 判类。

「几率刚好等于 0.5」的那条线，就是决策边界（decision boundary）：线的一边判为类别 1、另一边判为类别 0。当一条直线就能把两类大致分开时，称数据线性可分（linearly separable）。

为什么用图看：在二维散布图上把边界画出来，最能直觉看到模型是怎么「切」数据的。

In [ ]:
# 概念：两特征下，几率等于 0.5 的线即决策边界，把平面切成两类
rng = np.random.default_rng(0)   # 固定乱数种子，让每次结果一致

# 造两群带标签的点：类别 0（住宅，楼矮基地小）、类别 1（商办，楼高基地大）
res = rng.normal(loc=[15, 80], scale=[4, 15], size=(40, 2))    # 住宅：楼高 / 基地面积
com = rng.normal(loc=[45, 180], scale=[6, 25], size=(40, 2))   # 商办
X = np.vstack([res, com])                                      # 叠成 (80, 2) 特征矩阵
y = np.array([0] * 40 + [1] * 40)                              # 前 40 住宅、后 40 商办

# 自订一条决策边界（这里手动给定权重示意；下一节的损失与之后的工具会自动学）
w = np.array([0.08, 0.02])   # 两特征的权重
b = -5.0                     # 偏置 bias

# 边界是 w0*x0 + w1*x1 + b = 0，解出 x1 对 x0 的直线
x0_line = np.linspace(X[:, 0].min(), X[:, 0].max(), 50)
x1_line = -(w[0] * x0_line + b) / w[1]   # 由边界方程式移项求得

plt.figure(figsize=(5, 4))
plt.scatter(X[y == 0, 0], X[y == 0, 1], label="住宅 (0)")
plt.scatter(X[y == 1, 0], X[y == 1, 1], label="商办 (1)")
plt.plot(x0_line, x1_line, "k--", label="决策边界")
plt.xlabel("楼高")
plt.ylabel("基地面积")
plt.legend()
plt.title("决策边界把平面切成两类")
plt.show()

## 交叉熵损失的直觉

损失函数（loss function）衡量「预测有多差」，模型靠最小化它来学习。分类问题不用线性回归的均方误差，而用交叉熵（cross-entropy）。

公式（单笔）：loss = -[y·log(p) + (1-y)·log(1-p)]，其中 y 是真实标签、p 是模型给的几率。

直觉：交叉熵会放大「很有信心却猜错」的惩罚。当你断言几率 0.99 却其实是另一类，log 会给出很大的损失；猜对且有信心则损失趋近 0。这正是分类想要的：鼓励正确且校准（confidence）的几率。

In [ ]:
# 概念：用几组（真实标签, 预测几率）手算交叉熵，比较猜对与猜错的惩罚差距
def cross_entropy(y, p):
    eps = 1e-12                      # 注意：log(0) 会是负无限大，加极小值 eps 避免报错
    p = np.clip(p, eps, 1 - eps)     # 把几率夹在 (0, 1) 开区间内
    return -(y * np.log(p) + (1 - y) * np.log(1 - p))

# 每组是 (真实标签 y, 预测几率 p) 与情境说明
cases = [
    (1, 0.95, "猜对且有信心"),
    (1, 0.55, "猜对但没把握"),
    (1, 0.05, "猜错且很有信心"),
    (0, 0.02, "猜对且有信心"),
    (0, 0.97, "猜错且很有信心"),
]
for y, p, desc in cases:
    loss = cross_entropy(y, p)
    print(f"y={y}  p={p:.2f}  loss={loss:.3f}   ({desc})")

## 用准确率评估分类

准确率（accuracy）是最直觉的分类指针：猜对的笔数除以总笔数。

公式：accuracy = 猜对笔数 / 总笔数

它好懂，但有盲点：当类别不平衡（class imbalance）时会误导。例如 95% 的样本都是同一类，模型「全部猜多数类」也有 95% 准确率，看似很高却完全没抓到少数类。所以光看 accuracy 不够，要对不平衡保持警觉。

In [ ]:
# 概念：手算准确率，并用「全猜多数类」示范高准确率也可能没用
# 造一组不平衡的真实标签：18 笔类别 0、2 笔类别 1（少数类）
y_true = np.array([0] * 18 + [1] * 2)

# 模型 A：很懒，不管输入一律猜 0（多数类）
pred_all_zero = np.zeros_like(y_true)
acc_all_zero = (pred_all_zero == y_true).mean()   # 相等的比例即准确率

# 模型 B：有抓到一个少数类，但也误判一笔
pred_model_b = np.array([0] * 17 + [1] + [1, 0])
acc_model_b = (pred_model_b == y_true).mean()

print("全猜多数类 准确率:", round(acc_all_zero, 3))   # 高得吓人，却一个少数类都没抓到
print("模型 B    准确率:", round(acc_model_b, 3))
# 注意：两者准确率接近，但只有模型 B 真的抓到少数类；不平衡时别只看 accuracy
print("全猜多数类抓到的少数类数量:", int(((pred_all_zero == 1) & (y_true == 1)).sum()))
print("模型 B 抓到的少数类数量:", int(((pred_model_b == 1) & (y_true == 1)).sum()))

## sklearn LogisticRegression 实作示范

把前面的概念串起来：sklearn 的 LogisticRegression 会自动学出最适合的权重，最小化交叉熵。常用三个方法：

| 方法 | 作用 |
|---|---|
| fit(X, y) | 用训练数据学出模型参数 |
| predict(X) | 输出每笔的类别（0 / 1） |
| predict_proba(X) | 输出每笔属于各类的几率 |

实务上会先把数据切成训练集（train）与测试集（test），用没看过的测试集评估才公平；这里概念带过，重点在体会「自造数据到输出几率与类别」的完整流程。

In [ ]:
# 概念：用 LogisticRegression 跑完整流程，输出类别、几率与准确率
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(1)
# 造两类建筑数据：类别 0（住宅）、类别 1（商办），各 60 笔、两个特征
res = rng.normal(loc=[15, 80], scale=[4, 15], size=(60, 2))
com = rng.normal(loc=[45, 180], scale=[6, 25], size=(60, 2))
X = np.vstack([res, com])
y = np.array([0] * 60 + [1] * 60)

# train / test split：用测试集评估才不会「背答案」
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

model = LogisticRegression()
model.fit(X_train, y_train)          # 学出权重（自动最小化交叉熵）

pred = model.predict(X_test)                 # 每笔预测类别
proba = model.predict_proba(X_test)[:, 1]    # 取「属于类别 1」的几率（第 1 栏）
acc = (pred == y_test).mean()                # 测试集准确率

print("测试集准确率:", round(acc, 3))
print("前 5 笔  类别 / 属于商办的几率:")
for c, p in zip(pred[:5], proba[:5]):
    print(f"  类别 {c}   几率 {p:.3f}")

## 练习

以下三题由浅入深，皆为复合型但简短。数据请自己用 numpy 造（仿真即可），完成后对照「验收」确认输出。

In [ ]:
# TODO 1 ·（简单）高层建筑几率小判断（集成：sigmoid + 阈值）
#   情境：用「楼高」单一特征，判断一栋楼是否属于高层建筑（数据自造）。
#   要求：
#     1. 用 numpy 自造 10 笔楼高，并各算一个线性分数（例如 0.15 * 楼高 - 3.5）。
#     2. 套用 sigmoid 得到几率，再以 0.5 阈值转成 0 / 1 类别，逐笔印出。
#   验收：应看到每笔楼高对应的几率与最终类别，且楼越高几率越接近 1。
# TODO: 在这里完成

In [ ]:
# TODO 2 ·（中间）住宅与商办二分类（集成：决策边界 + LogisticRegression + 准确率）
#   情境：用「楼高 + 容积率」两个特征，把建物分成住宅与商办两类（数据自造）。
#   要求：
#     1. 用 numpy 自造两群带标签的二维数据（住宅一群、商办一群），用 LogisticRegression 训练。
#     2. 在散布图上画出两类点与一条决策边界，并计算整体准确率印出。
#   验收：应看到一条把两群大致分开的边界，以及一个明显高于乱猜（远大于 0.5）的准确率。
# TODO: 在这里完成

In [ ]:
# TODO 3 ·（稍难）阈值与交叉熵的权衡（集成：几率输出 + 交叉熵直觉 + 准确率 + 类别不平衡）
#   情境：在一份「是否为老旧待更新建物」的不平衡数据上做分类（多数为非待更新；数据自造）。
#   要求：
#     1. 自造一份正负模拟例失衡的数据（例如九成非待更新）并训练模型，取得每笔 predict_proba。
#     2. 用 0.3 / 0.5 / 0.7 三种阈值各自把几率转成类别，分别算准确率与「被判为正类的数量」。
#     3. 结合交叉熵直觉说明：为何在不平衡数据上「高准确率」未必代表模型抓到少数类。
#   验收：应看到不同阈值下准确率与正类数量的变化，并能讲出在不平衡情境下只看 accuracy 的盲点。
# TODO: 在这里完成

## 小结

- 分类问题的答案是类别（0 / 1），线性回归输出的任意实数无法直接当「是不是某一类」，需要能输出几率的模型。
- sigmoid 函数把线性分数压进 0 到 1，使输出可解读成几率；常用 0.5 当判类阈值。
- 两特征时，「几率刚好 0.5」的那条线就是决策边界，把特征空间切成两类；一条直线能分开即线性可分。
- 分类用交叉熵当损失，它对「很有信心却猜错」给很大惩罚，鼓励正确且校准的几率。
- 准确率是猜对的比例，直觉好懂，但在类别不平衡时会误导，光看 accuracy 不够。
- sklearn 的 LogisticRegression 用 fit / predict / predict_proba 就能完成从自造数据到输出几率与类别的完整流程。